# Week 3 — SQL Queries (Part A)
NextGenLearners Internship — Program Performance Analysis

This notebook explores the 3 related tables (`applicants`, `interns`, `hackathon_scores`) and answers 5 business questions using SQL.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nextgen.db')

## Step 2: Explore each table first

In [2]:
pd.read_sql("SELECT * FROM applicants LIMIT 5;", conn)

,applicant_id,name,domain,university,application_date,status
0,1,Applicant_1,Data Analytics,NED University,2025-01-07,Rejected
1,2,Applicant_2,Data Analytics,University of Karachi,2025-02-27,Selected
2,3,Applicant_3,Data Analytics,IBA Karachi,2025-06-23,Rejected
3,4,Applicant_4,Data Analytics,University of Karachi,2025-01-23,Rejected
4,5,Applicant_5,Data Analytics,NED University,2025-01-08,Selected


In [3]:
pd.read_sql("SELECT * FROM interns LIMIT 5;", conn)

,intern_id,applicant_id,domain,start_date,completion_status
0,1,2,Data Analytics,2025-03-22,Completed
1,2,5,Data Analytics,2025-02-03,Completed
2,3,9,Data Analytics,2025-06-28,Completed
3,4,12,Data Analytics,2025-02-20,Completed
4,5,13,Data Analytics,2025-04-07,Completed


In [4]:
pd.read_sql("SELECT * FROM hackathon_scores LIMIT 5;", conn)

,intern_id,score,domain
0,1,88.7,Data Analytics
1,2,71.7,Data Analytics
2,3,55.3,Data Analytics
3,4,70.5,Data Analytics
4,5,88.2,Data Analytics


## Query 1 — How many interns completed each domain's program?

In [5]:
query1 = """
-- Counts completed interns grouped by domain
SELECT domain, COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""
df1 = pd.read_sql(query1, conn)
df1

,domain,completed_count
0,Digital Marketing,15
1,Web Development,14
2,Data Analytics,11
3,UI/UX Design,8
4,AI/ML,7


**What's happening:** `GROUP BY domain` bundles rows with the same domain together, and `COUNT(*)` counts rows per bundle. The `WHERE` clause filters to completed interns only, so dropouts don't inflate the count. `ORDER BY ... DESC` puts the top domain first.

## Query 2 — What is the average hackathon score per domain?

In [6]:
query2 = """
-- Calculates the average hackathon score for each domain
SELECT domain, ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""
df2 = pd.read_sql(query2, conn)
df2

,domain,avg_score
0,Digital Marketing,80.69
1,Data Analytics,80.53
2,Web Development,69.76
3,UI/UX Design,67.30
4,AI/ML,67.27


**What's happening:** `AVG(score)` computes the mean score per domain group. `ROUND(..., 2)` keeps the result to 2 decimal places for readability.

## Query 3 — Which interns scored above a threshold (top performers)?

In [7]:
query3 = """
-- Lists interns who scored 85 or above, as candidates for showcase/certificates
SELECT i.intern_id, i.domain, h.score
FROM interns i
JOIN hackathon_scores h ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""
df3 = pd.read_sql(query3, conn)
df3

,intern_id,domain,score
0,48,Digital Marketing,99.0
1,7,Data Analytics,97.0
2,55,Digital Marketing,96.8
3,6,Data Analytics,92.6
4,43,Digital Marketing,91.6
5,52,Digital Marketing,89.4
6,1,Data Analytics,88.7
7,40,Digital Marketing,88.5
8,5,Data Analytics,88.2
9,54,Digital Marketing,88.2


**What's happening (first JOIN):** `JOIN ... ON i.intern_id = h.intern_id` matches rows from both tables wherever `intern_id` matches. `WHERE h.score >= 85` filters to top performers.

## Query 4 — Conversion rate from 'applied' to 'completed' per domain

In [8]:
query4 = """
-- Compares total applicants vs completed interns per domain to find conversion rate
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) / COUNT(DISTINCT a.applicant_id), 2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
    ON a.applicant_id = i.applicant_id AND i.completion_status = 'Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""
df4 = pd.read_sql(query4, conn)
df4

,domain,total_applicants,total_completed,conversion_rate_pct
0,Web Development,33,14,42.42
1,Digital Marketing,38,15,39.47
2,Data Analytics,28,11,39.29
3,UI/UX Design,29,8,27.59
4,AI/ML,33,7,21.21


**What's happening (first LEFT JOIN):** `LEFT JOIN` keeps every applicant row, even those with no matching completed intern. This matters — a regular `JOIN` would only keep applicants who *did* convert, making the conversion rate meaningless (always 100%). `COUNT(DISTINCT ...)` avoids double-counting, and `100.0 *` forces floating-point (decimal) division instead of integer division.

## Query 5 — Your Choice: How many interns dropped out vs completed, per domain?

In [9]:
query5 = """
-- Compares dropout vs completion counts per domain to spot at-risk programs
SELECT
    domain,
    completion_status,
    COUNT(*) AS intern_count
FROM interns
GROUP BY domain, completion_status
ORDER BY domain, completion_status;
"""
df5 = pd.read_sql(query5, conn)
df5

,domain,completion_status,intern_count
0,AI/ML,Completed,7
1,AI/ML,Dropped Out,3
2,Data Analytics,Completed,11
3,Data Analytics,Dropped Out,1
4,Digital Marketing,Completed,15
5,Digital Marketing,Dropped Out,1
6,UI/UX Design,Completed,8
7,UI/UX Design,Dropped Out,5
8,Web Development,Completed,14


**What's happening:** `GROUP BY domain, completion_status` creates a bundle for every domain + status combination, and `COUNT(*)` counts interns in each. This shows, per domain, how many interns completed vs. dropped out — useful for spotting which programs need more support to retain interns.

## Wrap up

In [10]:
conn.close()